# Algoritmo genético para N reinas con DEAP

Cada individuo es una permutación: el índice es la columna y el valor es la fila.

## Dependencias

En un entorno nuevo: `%pip install deap numpy matplotlib`.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from deap import algorithms, base, creator, tools

In [ ]:
N = 12

def conflictos(individuo):
    total = sum(
        abs(individuo[i] - individuo[j]) == abs(i - j)
        for i in range(len(individuo))
        for j in range(i + 1, len(individuo))
    )
    return (total,)

## Toolbox

Los nombres de `creator` se protegen para que el notebook pueda volver a ejecutarse en el mismo kernel.

In [ ]:
if not hasattr(creator, 'FitnessNReinasMin'):
    creator.create('FitnessNReinasMin', base.Fitness, weights=(-1.0,))
if not hasattr(creator, 'IndividuoNReinas'):
    creator.create('IndividuoNReinas', list, fitness=creator.FitnessNReinasMin)

toolbox = base.Toolbox()
toolbox.register('indices', random.sample, range(N), N)
toolbox.register('individual', tools.initIterate, creator.IndividuoNReinas, toolbox.indices)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('evaluate', conflictos)
toolbox.register('select', tools.selTournament, tournsize=3)
toolbox.register('mate', tools.cxOrdered)
toolbox.register('mutate', tools.mutShuffleIndexes, indpb=2 / N)

In [ ]:
def ejecutar(semilla, n_poblacion=160, generaciones=180):
    random.seed(semilla)
    np.random.seed(semilla)
    poblacion = toolbox.population(n=n_poblacion)
    elite = tools.HallOfFame(1)
    estadisticas = tools.Statistics(lambda ind: ind.fitness.values[0])
    estadisticas.register('min', np.min)
    estadisticas.register('avg', np.mean)
    _, log = algorithms.eaSimple(
        poblacion, toolbox, cxpb=0.8, mutpb=0.25, ngen=generaciones,
        stats=estadisticas, halloffame=elite, verbose=False,
    )
    return elite[0], log

mejor, log = ejecutar(semilla=2026)
print('Solución:', mejor)
print('Conflictos:', conflictos(mejor)[0])

In [ ]:
generaciones = log.select('gen')
plt.plot(generaciones, log.select('avg'), label='promedio')
plt.plot(generaciones, log.select('min'), label='mejor')
plt.xlabel('Generación')
plt.ylabel('Conflictos diagonales')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Variabilidad entre ejecuciones

In [ ]:
resultados = []
for semilla in range(10):
    individuo, _ = ejecutar(semilla, generaciones=100)
    resultados.append(conflictos(individuo)[0])

print('Conflictos finales:', resultados)
print('Tasa de éxito:', np.mean(np.asarray(resultados) == 0))

## Trabajo propuesto

Compare torneo de tamaños 2, 3 y 7 con igual presupuesto. Reporte tasa de éxito, mediana de conflictos y diversidad de la población.